# Capstone Mini-Project: Energy-Demand Forecasting with LSTM + Attention

This notebook builds an end-to-end 24-hour energy demand forecaster from a self-generated hourly series. It compares a seasonal-naive baseline, a direct LSTM, and an LSTM with self-attention.

**Goal:** use the previous 168 hours (1 week) to predict the next 24 hours (1 day), then evaluate with MAE, RMSE, and MAPE.


## 0. Imports and synthetic energy-demand series
The series contains a slow trend, daily seasonality, weekly seasonality, a weekend dip, and Gaussian noise. Everything is generated offline.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RNG = 42
np.random.seed(RNG)
tf.random.set_seed(RNG)

N = 24 * 7 * 104
t = np.arange(N)
trend = 0.0015 * t
daily = 8.0 * np.sin(2 * np.pi * t / 24 - 1.0)
weekly = 4.0 * np.sin(2 * np.pi * t / (24 * 7))
weekend = -3.0 * (((t // 24) % 7) >= 5)
noise = np.random.normal(0, 1.2, N)

demand = 50.0 + trend + daily + weekly + weekend + noise
demand = demand.astype('float32')

print('series length:', demand.shape)
print('mean:', round(float(demand.mean()), 2))
print('min:', round(float(demand.min()), 2))
print('max:', round(float(demand.max()), 2))


## 1. Sliding windows: 168 hours → 24 hours
Each input contains one week of hourly history. Its target is the immediately following 24 hours. The window moves one hour at a time.


In [ ]:
LOOKBACK = 168
HORIZON = 24

def make_windows(series, lookback, horizon):
    X, y = [], []
    for i in range(len(series) - lookback - horizon + 1):
        X.append(series[i:i + lookback])
        y.append(series[i + lookback:i + lookback + horizon])
    return np.array(X, dtype='float32'), np.array(y, dtype='float32')

X_all, y_all = make_windows(demand, LOOKBACK, HORIZON)
print('X_all:', X_all.shape, 'y_all:', y_all.shape)
print('Expected samples:', len(demand) - LOOKBACK - HORIZON + 1)


## 2. Chronological train/validation/test split and train-only normalization
We do not shuffle a time series. The windows remain in time order, with 70% used for training, 15% for validation, and 15% for final testing. Normalization statistics are computed from training inputs only.


In [ ]:
n = len(X_all)
i_tr = int(n * 0.70)
i_va = int(n * 0.85)

X_tr_raw, y_tr_raw = X_all[:i_tr], y_all[:i_tr]
X_va_raw, y_va_raw = X_all[i_tr:i_va], y_all[i_tr:i_va]
X_te_raw, y_te_raw = X_all[i_va:], y_all[i_va:]

mu = X_tr_raw.mean()
sigma = X_tr_raw.std()

def norm(a):
    return (a - mu) / sigma

def denorm(a):
    return a * sigma + mu

X_tr, y_tr = norm(X_tr_raw), norm(y_tr_raw)
X_va, y_va = norm(X_va_raw), norm(y_va_raw)
X_te, y_te = norm(X_te_raw), norm(y_te_raw)

# Keras RNN input shape: (samples, timesteps, features)
X_tr = X_tr[..., None]
X_va = X_va[..., None]
X_te = X_te[..., None]

print('train:', X_tr.shape, y_tr.shape)
print('val:  ', X_va.shape, y_va.shape)
print('test: ', X_te.shape, y_te.shape)
print('mu:', round(float(mu), 3), 'sigma:', round(float(sigma), 3))


### Leakage note
The split is chronological and normalization uses training statistics only. Because the exercise creates overlapping windows before splitting, adjacent boundary windows can share historical timestamps. The targets still progress chronologically. For a stricter production experiment, split the raw series first and construct windows within each partition (or explicitly carry only permitted past context into validation/test).


## 3. Seasonal-naive baseline
The baseline predicts that the next 24 hours will look like the last 24 hours in the input window. This is a strong baseline because the generated series has a clear 24-hour cycle.


In [ ]:
def mae(a, b):
    return float(np.mean(np.abs(a - b)))

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def mape(a, b):
    return float(np.mean(np.abs((a - b) / a)) * 100)

baseline_pred = X_te_raw[:, -HORIZON:]
y_te_true = y_te_raw

b_mae = mae(y_te_true, baseline_pred)
b_rmse = rmse(y_te_true, baseline_pred)
b_mape = mape(y_te_true, baseline_pred)

print(f'SEASONAL-NAIVE MAE={b_mae:.3f} RMSE={b_rmse:.3f} MAPE={b_mape:.2f}%')


## 4. Plain LSTM forecaster
The LSTM reads all 168 input hours and returns its final hidden state. A Dense layer predicts all 24 future steps at once, so this is **direct multi-output forecasting**.


In [ ]:
def build_lstm():
    inp = keras.Input(shape=(LOOKBACK, 1))
    x = layers.LSTM(48)(inp)
    x = layers.Dropout(0.1)(x)
    out = layers.Dense(HORIZON)(x)
    model = keras.Model(inp, out, name='lstm_direct')
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='mse',
        metrics=['mae']
    )
    return model

lstm = build_lstm()
lstm.summary()

es = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
hist = lstm.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=15,
    batch_size=128,
    callbacks=[es],
    verbose=2
)

print('epochs run:', len(hist.history['loss']))
print('best val_mae(norm):', round(min(hist.history['val_mae']), 4))


## 5. LSTM with attention over encoder states
Here `return_sequences=True` preserves the LSTM state at every input hour. Self-attention lets states interact before global pooling creates the representation used to predict the next 24 hours.


In [ ]:
def build_lstm_attention():
    inp = keras.Input(shape=(LOOKBACK, 1))
    enc = layers.LSTM(48, return_sequences=True)(inp)
    ctx = layers.Attention()([enc, enc])
    ctx = layers.LayerNormalization()(ctx + enc)
    pooled = layers.GlobalAveragePooling1D()(ctx)
    x = layers.Dropout(0.1)(pooled)
    out = layers.Dense(HORIZON)(x)
    model = keras.Model(inp, out, name='lstm_attention')
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='mse',
        metrics=['mae']
    )
    return model

lstm_attn = build_lstm_attention()
lstm_attn.summary()

es2 = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
hist2 = lstm_attn.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=15,
    batch_size=128,
    callbacks=[es2],
    verbose=2
)

print('epochs run:', len(hist2.history['loss']))
print('best val_mae(norm):', round(min(hist2.history['val_mae']), 4))
print(lstm_attn.count_params(), 'params vs', lstm.count_params(), 'for plain LSTM')


## 6. Test evaluation in original demand units
Predictions are converted back from normalized values before evaluation. Lower MAE, RMSE, and MAPE are better.


In [ ]:
def score_model(model, name):
    pred = denorm(model.predict(X_te, verbose=0))
    return name, mae(y_te_true, pred), rmse(y_te_true, pred), mape(y_te_true, pred)

rows = [
    ('seasonal-naive', b_mae, b_rmse, b_mape),
    score_model(lstm, 'lstm'),
    score_model(lstm_attn, 'lstm+attention'),
]

print(f"{'model':16s} {'MAE':>8s} {'RMSE':>8s} {'MAPE%':>8s}")
for name, a, r, p in rows:
    print(f'{name:16s} {a:8.3f} {r:8.3f} {p:8.2f}')

best = min(rows, key=lambda z: z[1])
print('
lowest MAE:', best[0])

print('
Deltas vs baseline (negative means lower error than baseline)')
for name, a, r, p in rows[1:]:
    print(f'{name:16s} dMAE={a-b_mae:+.3f} dRMSE={r-b_rmse:+.3f} dMAPE={p-b_mape:+.2f} pp')


## 7. One 24-hour forecast and inference discussion


In [ ]:
k = 0
truth = y_te_true[k]
naive = baseline_pred[k]

# Plot the model with the lower test MAE among the two neural networks.
lstm_row = next(row for row in rows if row[0] == 'lstm')
attn_row = next(row for row in rows if row[0] == 'lstm+attention')
if attn_row[1] <= lstm_row[1]:
    plot_model = lstm_attn
    plot_name = 'lstm+attention'
else:
    plot_model = lstm
    plot_name = 'lstm'

net = denorm(plot_model.predict(X_te[k:k+1], verbose=0))[0]

plt.figure(figsize=(7, 4))
plt.plot(truth, 'o-', label='truth')
plt.plot(naive, 's--', label='seasonal-naive')
plt.plot(net, '^--', label=plot_name)
plt.xlabel('hours ahead (1..24)')
plt.ylabel('demand')
plt.title('24-hour-ahead forecast: one test window')
plt.legend()
plt.tight_layout()
plt.show()

print('truth[:6]:', np.round(truth[:6], 2))
print('naive[:6]:', np.round(naive[:6], 2))
print('net[:6]:  ', np.round(net[:6], 2))


### Plot interpretation and teacher forcing
The selected neural-network forecast should follow the general daily rise-and-fall pattern of the true demand. The seasonal-naive line simply repeats the previous day's shape, whereas the learned network can also use longer weekly context and the trend. After running the cell above, compare the exact curves and edit this paragraph if your result shows a different pattern.

Both networks in this notebook predict all 24 future steps simultaneously through a Dense output layer. Therefore, they use **direct multi-output forecasting**, not teacher forcing or autoregressive inference. In a traditional seq2seq decoder trained with teacher forcing, each decoder step receives the ground-truth previous value during training, but at inference it must use its own previous prediction. That train/inference mismatch is called **exposure bias**, and errors can compound across forecast steps.


## 8. Analysis

**Run Step 6 first, then replace every bracketed value below with your actual output. Do not submit invented numbers.**

The seasonal-naive baseline achieved an MAE of **[baseline MAE]**, an RMSE of **[baseline RMSE]**, and a MAPE of **[baseline MAPE]%**. The plain LSTM achieved an MAE of **[LSTM MAE]**, an RMSE of **[LSTM RMSE]**, and a MAPE of **[LSTM MAPE]%**. Relative to the baseline, its errors changed by **[dMAE] MAE**, **[dRMSE] RMSE**, and **[dMAPE] percentage points MAPE**. The LSTM-with-attention model achieved an MAE of **[attention MAE]**, an RMSE of **[attention RMSE]**, and a MAPE of **[attention MAPE]%**.

Based on the test results, **[best model]** produced the lowest MAE. Attention **[did/did not]** justify its added complexity because its improvement over the plain LSTM was **[substantial/small/no improvement]**. Attention would likely matter more for longer sequences and horizons, irregular seasonal patterns, or exogenous drivers such as temperature, holidays, and electricity prices.

The data was kept in chronological order, and normalization statistics were fitted only on training inputs. The test set was reserved for final evaluation. Both neural networks predict all 24 steps directly, so teacher forcing is not used. An autoregressive decoder could suffer from exposure bias because early prediction errors may propagate to later steps. A useful next step would be to add hour-of-day and day-of-week features or build a true seq2seq decoder.


### Final metric values to copy into the analysis


In [ ]:
print('BASELINE MAE=%.3f RMSE=%.3f MAPE=%.2f%%' % (b_mae, b_rmse, b_mape))
for name, a, r, p in rows[1:]:
    print('%-16s MAE=%.3f RMSE=%.3f MAPE=%.2f%% (dMAE=%+.3f, dRMSE=%+.3f, dMAPE=%+.2f pp)' %
          (name, a, r, p, a-b_mae, r-b_rmse, p-b_mape))
